# 21-review-analyses, reviewer analyses on CPU

Settings: **Accelerator None (CPU)**, **Internet On**, secret `kaggle_dpfl-braintumor` attached. No input dataset is needed.

Runs code commit `0ff16425f164b79b3cd4efa873810a366340fea3`: tests, then LiRA with source- and class-matched non-members, per-class sensitivity on the clean test set, and the explanation statistics at model and configuration level, all from the run and audit files on GitHub. Pushes the tables to `results/tables/`. Uses no GPU quota.

Generated by `scripts/make_launcher.py --kind review`, edit that script rather than this notebook.

In [ ]:
import os, sys, glob, subprocess, datetime, shutil
from kaggle_secrets import UserSecretsClient

USER, EMAIL, REPO = "shashiranjan-jnu", "shashi17_scs@jnu.ac.in", "dpfl-braintumor"
CODE_REF = "0ff16425f164b79b3cd4efa873810a366340fea3"
token = UserSecretsClient().get_secret("kaggle_dpfl-braintumor")
os.environ["GIT_PUSH_URL"] = f"https://{token}@github.com/{USER}/{REPO}.git"
CLEAN = f"https://github.com/{USER}/{REPO}.git"
CODE, RES = "/tmp/code", "/tmp/results"

def run(cmd, cwd=None, check=True):
    r = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    out = (r.stdout + r.stderr).replace(token, "***")
    if check and r.returncode != 0:
        raise RuntimeError(out[-3000:])
    return out

for d in (CODE, RES):
    shutil.rmtree(d, ignore_errors=True)
    run(["git", "clone", "-q", os.environ["GIT_PUSH_URL"], d])
    run(["git", "remote", "set-url", "origin", CLEAN], cwd=d)
run(["git", "checkout", "-q", CODE_REF], cwd=CODE)
run(["git", "config", "user.name", USER], cwd=RES)
run(["git", "config", "user.email", EMAIL], cwd=RES)
sys.path.insert(0, CODE)
from scripts.run_grid import push_run


print(run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{CODE}/requirements.txt"])[-500:])  # opacus etc.
import torch
TODAY = datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%d")
COMMIT = run(["git", "rev-parse", "HEAD"], cwd=CODE).strip()
for f in ("results/ncc/curated_pairs_ncc090.csv", "results/tables/week05_h5_mechanism.csv",
          "configs/grids/week07_wn_seeds.yaml"):
    assert os.path.exists(f"{RES}/{f}"), f"missing {f} in the results repository"
n_logits = len(glob.glob(f"{RES}/results/audit/*/logits.npz"))
print("code", CODE_REF, run(["git", "rev-parse", "--short", "HEAD"], cwd=CODE).strip(), "| logits files", n_logits)

In [ ]:
# Test suite on the Kaggle image, report pushed to results/tests/.
sys.path.insert(0, CODE)
from scripts.run_grid import push_run
r = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests", "-p", "no:cacheprovider"],
                   cwd=CODE, capture_output=True, text=True)
report = f"date {TODAY}\ncode {CODE_REF} {COMMIT}\ntorch {torch.__version__}\nreturncode {r.returncode}\n\n" + r.stdout[-6000:]
print(report[-2500:])
os.makedirs(f"{RES}/results/tests", exist_ok=True)
open(f"{RES}/results/tests/{TODAY}_{CODE_REF}_pytest.txt", "w").write(report)
print("push", push_run(RES, "results/tests", f"Test report {CODE_REF}"))
assert r.returncode == 0, "Tests failed, stop here and report"

In [ ]:
# Reviewer analyses (D59), CPU only, a few minutes. No training and no images are read.
R = f"{RES}/results"
T = f"{R}/tables"
print(run([sys.executable, "scripts/mia_matched.py", "--results", R, "--grids-dir", f"{RES}/configs/grids",
           "--exclude", f"{R}/ncc/contaminated_test_ids.txt", "--out", f"{T}/week07_mia_matched"], cwd=CODE)[-6000:])
print(run([sys.executable, "scripts/class_sensitivity.py", "--results", R,
           "--out", f"{T}/week07_class_sensitivity"], cwd=CODE)[-4000:])
print(run([sys.executable, "scripts/xai_mediation.py", "--results", R, "--out", f"{T}/week07_xai_mediation"], cwd=CODE))
print("push tables", push_run(RES, "results/tables", f"Reviewer analyses on Kaggle ({CODE_REF})"))